Author: Rajas Joshi

Note: Much of the following work uses code that I've written in the various python files in the folder "numi_libs". Any '.py' files attached in my solution should be considered a part of that library; the questions themselves are in '.ipynb' files (ie notebooks).

To run this code successfully, create a folder called "numi_libs" in the same directory and move all '.py' files there.

Question #

In [ ]:
import pandas as pd
import numpy as np
import warnings

from sklearn.linear_model import LinearRegression, LogisticRegression, SGDClassifier
from sklearn.svm import SVR, SVC
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier

import os
import sys

sys.path.append(os.getcwd() + '/numi_libs/')

import file_io as fio
import data_eda as eda
import data_forge as frg
import data_model as dmd
import data_docs as dds

warnings.filterwarnings('ignore')

seed = 420
np.random.seed(seed)

In [ ]:
filename_c = "patient-data.csv"
filename_n = 'curse-of-dimensionality.xlsx'

df_c = fio.readCsvToDF(filename_c)

eda.initialAssessment(df_c, "Patient Dataset")


file_path = 'curse-of-dimensionality.xlsx'
df_n = pd.read_excel(filename_n, sheet_name='Sheet3')

eda.initialAssessment(df_n, "Curse of Dimensionality Dataset")
print("assessment complete")

In [ ]:
eda.profileFeatures(df_n)

In [ ]:
x_tr, x_te, y_tr, y_te = frg.ttSplit(df_n, "y")
x_tr_trf, x_te_trf, _= frg.yjTransform(x_tr, x_te)
x_tr_trf, x_te_trf, _= frg.minMaxScale(x_tr_trf, x_te_trf)
eda.profileFeatures(x_tr_trf)

In [ ]:
algorithms = {
    'Linear Regression': LinearRegression(),
    #'SVM Regression': SVR(kernel='linear'),  # Adjust kernel as needed
    #'RandomForest': RandomForestRegressor(),
    #'XGBoost': GradientBoostingRegressor(),
    #'knn': KNeighborsRegressor(),
    #'Neural Network-10-5-5': MLPRegressor(hidden_layer_sizes=[10, 5, 5], max_iter=20000),
}

pca = [0]

dmd.applyAndReportRegressors(x_tr, x_te, y_tr, y_te, algorithms, pca, False, "Vanilla")
dmd.applyAndReportRegressors(x_tr_trf, x_te_trf, y_tr, y_te, algorithms, pca, False, "Transformed")

In [ ]:
vifs = eda.probeVifRemoval(df_n.drop(columns = ['y']))

print(vifs)

In [ ]:
x_tr_trf_vif = x_tr_trf.drop(columns = ['x1', 'x8', 'x5', 'x3', 'x7'])
x_te_trf_vif = x_te_trf.drop(columns = ['x1', 'x8', 'x5', 'x3', 'x7'])
dmd.applyAndReportRegressors(x_tr_trf_vif, x_te_trf_vif, y_tr, y_te, algorithms, pca, True, "Transformed + VIF")


In [ ]:
eda.performPcaAnalysis(x_tr_trf)

In [ ]:
eda.performPcaAnalysis(x_tr)

In [ ]:
pca = [0, 2]
dmd.applyAndReportRegressors(x_tr_trf, x_te_trf, y_tr, y_te, 
                             algorithms, pca, False, "Transformed")

In [ ]:
filename_c = "patient-data.csv"
filename_n = 'curse-of-dimensionality.xlsx'

df_c = fio.readCsvToDF(filename_c)

eda.initialAssessment(df_c, "Patient Dataset")


file_path = 'curse-of-dimensionality.xlsx'
df_n = pd.read_excel(filename_n, sheet_name='Sheet3')

eda.initialAssessment(df_n, "Curse of Dimensionality Dataset")
print("assessment complete")

In [ ]:
eda.generateEDATables(df_c)

In [ ]:
df_c = frg.simpleImputation(df_c)

x = df_c.drop(columns = ['Ailment'])
y = df_c['Ailment']

x, y = frg.smoteennSample(x, y, {
    "Heart Disease": 500,
    "Thromboc": 500
})

"""
x, y = frg.tomekSample(x, y, [
    "Heart Disease",
    "Thromboc"
])
"""

df_c0 = x
df_c0["Ailment"] = y
eda.generateEDATables(df_c0)

In [ ]:
eda.performPcaAnalysis(df_c0.drop(columns = ['Ailment']))

In [ ]:
algorithms = {
    #'Linear Classifier': SVC(kernel = "linear", probability = True),
    'Logistic Regression': LogisticRegression(max_iter = 2000), 
    #'RandomForest': RandomForestClassifier(max_depth = 5, min_samples_leaf = 5),
    #'XGBoost': GradientBoostingClassifier(),
    #'knn': KNeighborsClassifier(),
    #'Neural Network-10, 10': MLPClassifier(hidden_layer_sizes=[10, 10], max_iter=5000),
}

pca = []

x_tr, x_te, y_tr, y_te = frg.ttSplit(df_c, "Ailment", True)

dmd.applyAndReportClassifiers(x_tr, x_te, y_tr, y_te, algorithms, pca)

In [ ]:
x = df_c.drop(columns = ['Ailment'])
y = df_c['Ailment']

x, y = frg.smoteennSample(x, y, {
    "Heart Disease": 500,
    "Thromboc": 500
})

"""
x, y = frg.tomekSample(x, y, [
    "Heart Disease",
    "Thromboc"
])
"""

df_c0 = x
df_c0["Ailment"] = y

x_tr, x_te, y_tr, y_te = frg.ttSplit(df_c0, "Ailment", True)

dmd.applyAndReportClassifiers(x_tr, x_te, y_tr, y_te, algorithms, pca)

In [ ]:
x_tr, x_te, _= frg.minMaxScale(x_tr, x_te)

dmd.applyAndReportClassifiers(x_tr, x_te, y_tr, y_te, algorithms, pca, "MinMaxed")

In [ ]:
filename_m = "mnist_test_nolabels.csv"

df_m = fio.readCsvToDF(filename_m)

eda.initialAssessment(df_m, "Mnist Dataset")

In [ ]:
eda.performPcaAnalysis(df_m)

In [ ]:
dmd.performTSNE(df_m)

In [ ]:
filename_d = "Session-Summary-all-2025-S1.csv"

df_d = fio.readCsvToDF(filename_d)

eda.initialAssessment(df_d, "Mnist Dataset")

In [ ]:
df = dds.preprocessTextDF(df_d)

In [ ]:
matrices = dict()
df, matrices = dds.vectorizeDocumentDF(df, matrices)
df, matrices = dds.word2VecDocumentDF(df, matrices)
dds.computeVectorizerDistances(df, matrices)